# Kolokvijum I (Septembar 1) — Glavna knjiga

Ovaj notebook objašnjava rešenje iz **`k1_provere_na_kraju.html`**: koje koncepte funkcionalnog
programiranja zadatak traži, kojim mehanizmima su ostvareni i po čemu se to razlikuje od objektnog pristupa.

## Tekst zadatka

> Napraviti konstruktor tipa **GlavnaKnjiga** koji u sebi sadrži podatke naziv preduzeća, matični broj, pib i transakcije.
> Atribut transakcije je niz **Transakcija** objekata koji u sebi sadrže redni broj, broj računa, opis transakcije,
> status transakcije, datum, tip i iznos.
> **Redni broj transakcije je privatni statički atribut** čija vrednost se uvećava za jedan prilikom instanciranja svake transakcije.
> Status transakcije je string čije vrednosti mogu biti `"nerealizovana"`, `"realizovana"` i `"stornirana"`.
> Tip transakcije je string čije vrednosti mogu biti `"na teret"` i `"u korist"`.
>
> Glavna knjiga sadrži metode **dodajTransakciju** i **ukloniTransakciju**. Ove metode **ne smeju da mutiraju**
> originalni objekat nad kojim su primenjene. Rezultat ovih metoda je **nova instanca** glavne knjige u koju je dodata
> ili iz koje je uklonjena navedena transakcija. Novodobijena glavna knjiga sadrži i celokupnu **istoriju izmena**
> u vidu **reference na prethodnu glavnu knjigu** od koje je nastala.
> U glavnoj knjizi dodati **izvedeni atribut stanje** koji predstavlja razliku suma realizovanih transakcija u korist i na teret.
>
> Napraviti funkciju **pretraga** koja prima niz kriterijuma i glavnu knjigu, a kao rezultat vraća niz filtriran
> na osnovu kriterijuma. Kriterijumi su proizvoljno definisane predikatske funkcije. Primena kriterijuma pretrage
> se vrši onim redom kojim su navedeni u argumentima funkcije pretraga.
>
> Definisati funkciju **opoziv**, ova funkcija kao argumente prima glavnu knjigu i redni broj glavne knjige u istoriji.
> Rezultat funkcije je nova glavna knjiga koja u sebi sadrži sve transakcije koje je sadržala glavna knjiga na zadatom
> indeksu u istoriji, kao i sve transakcije koje su nastale nakon te glavne knjige, uz izmenu da su sve transakcije
> nastale nakon te glavne knjige statusa **stornirano**.
> Ova funkcija ne sme da menja nijednu od glavnih knjiga niti neku od njihovih transakcija.
>
> Zarad testiranja napraviti nekoliko instanci tipa GlavnaKnjiga, pozvati metode i funkcije nad njima i ispisati rezultate.

## Podela na logičke jedinice

| Jedinica | Šta sadrži | Glavni mehanizam | Koncept FP-a |
| --- | --- | --- | --- |
| 1 | `Transakcija`, dozvoljene vrednosti | IIFE + zatvorenje, `Object.freeze` | nepromenljivost, skriveno stanje na jednom mestu |
| 2 | čiste funkcije nad nizom transakcija | `spread`, `filter`, `reduce` | čista funkcija, preklapanje umesto petlje |
| 3 | `GlavnaKnjiga` | fabrička funkcija, `Object.defineProperties` + `get` | trajna struktura, izvedena vrednost |
| 4 | `istorija` | rekurzija nad lancem `prethodna` | deklarativni opis umesto postupka |
| 5 | `pretraga` | funkcije višeg reda, `reduce` nad predikatima | funkcije kao vrednosti, parcijalna primena |
| 6 | `opoziv` | rad nad nizom verzija, `map` sa kopijom | referencijalna transparentnost |
| P | provere | poređenje dobijenog i očekivanog | testiranje bez nameštanja stanja |

## Raspored rešenja

Rešenje je podeljeno na dva dela, i notebook prati taj isti redosled:

- **DEO I — IMPLEMENTACIJA** · samo definicije, nijedan ispis. Kod se čita u komadu.
- **DEO II — PROVERE** · instanciranje, pozivi i sav ispis, na kraju, sa završnim izveštajem `N/N provera prošlo`.

To razdvajanje nije kozmetičko — moguće je **zato što su sve funkcije čiste**. Kada rezultat zavisi samo od
argumenata, svejedno je da li se poziv izvršava odmah po definiciji ili tek na kraju fajla. U kodu sa promenljivim
stanjem to ne bi važilo: tamo redosled poziva menja rezultat, pa se provera mora naslanjati na tačno mesto u toku programa.

Odluke koje tekst zadatka ne propisuje, a moraju se doneti, označene su sa **Pretpostavka**.

## Pravila funkcionalne paradigme kojih se rešenje drži

| Pravilo | Kako je sprovedeno u ovom rešenju |
| --- | --- |
| **nepromenljivost** | `Object.freeze` nad svakom transakcijom, nad nizom transakcija i nad knjigom; nigde `push`, `splice` ni dodela postojećem polju |
| **bez `this` i bez `new`** | fabričke funkcije koje **vraćaju** objekat; sve metode su strelice nad zatvorenjem, pa ne mogu da izgube kontekst |
| **objekat se gradi jednim izrazom** | `Object.freeze(Object.defineProperties({ … }, { … }))` — nema koraka u kome objekat postoji nedovršen |
| **čiste funkcije** | `sa`, `bez`, `stanjeOd`, `stornirana`, `istorija`, `pretraga` i `opoziv` zavise samo od argumenata i ništa ne menjaju |
| **funkcije su vrednosti** | kriterijumi se prosleđuju kao argumenti; `preko(granica)` i `uMesecu(mesec)` **vraćaju** funkciju |
| **preklapanje umesto petlje** | `reduce`, `map`, `filter` i rekurzija; u celom rešenju nema `for` petlje ni brojača |
| **bez deljenog promenljivog stanja** | svaka izmena daje novu verziju; stare verzije ostaju važeće i dele iste zamrznute transakcije |
| **efekti na ivici** | `console.log` postoji isključivo u DEO II; nijedna funkcija koja računa rezultat ne ispisuje ništa |

### Dva svesna izuzetka

**Brojač rednog broja.** „Privatni statički atribut koji se uvećava pri svakom instanciranju" je po definiciji stanje,
pa `Transakcija` **nije** referencijalno transparentna — dva poziva sa istim argumentima daju različit redni broj.
To traži sam tekst zadatka. Stanje je zatvoreno u jednoj promenljivoj unutar IIFE-a (jedinica 1) i nigde se više ne pojavljuje.

**Bacanje izuzetka.** Provera dozvoljenih vrednosti (`status`, `tip`) i nepostojećeg indeksa u `opoziv` rešena je
sa `throw`. To je jedini nelokalni izlaz u rešenju; potpuno funkcionalna zamena bila bi povratna vrednost tipa
„uspeh ili greška" (`Either`), što tekst zadatka ne traži.

---
# DEO I — IMPLEMENTACIJA

Ćelije ispod su **samo definicije**. Nijedna ništa ne ispisuje i nijedna ne pravi test podatke —
to dolazi tek u DEO II.

Jedini izuzetak je **Digresija** posle Jedinice 2: samostalan primer koji ne pripada rešenju, nego objašnjava
po kom pravilu je odlučeno šta ide unutar tipa, a šta van njega.

---
# Jedinica 1 · `Transakcija` i privatni statički redni broj

In [ ]:
const STATUSI = Object.freeze(["nerealizovana", "realizovana", "stornirana"]);
const TIPOVI  = Object.freeze(["na teret", "u korist"]);

// IIFE napravi promenljivu i odmah vrati fabriku koja je jedina vidi.
// Posle ovog izraza ne postoji nijedan način da se do brojača dođe spolja.
const Transakcija = (() => {
    let sledeciRedniBroj = 1;                       // privatni statički atribut

    return (brojRacuna, opis, status, datum, tip, iznos) => {
        if (!STATUSI.includes(status)) throw new Error(`Nedozvoljen status: ${status}`);
        if (!TIPOVI.includes(tip))     throw new Error(`Nedozvoljen tip: ${tip}`);

        return Object.freeze({
            redniBroj: sledeciRedniBroj++,          // uvećava se za jedan pri svakom instanciranju
            brojRacuna,                             // broj računa
            opis,                                   // opis transakcije
            status,                                 // status transakcije
            datum,
            tip,
            iznos
        });
    };
})();

### Koncepti

**Zatvorenje (closure).** Funkcija pamti okruženje u kome je nastala. Strelica koju IIFE vraća i dalje vidi
`sledeciRedniBroj` iako je spoljna funkcija odavno završila. To je jedini mehanizam privatnosti koji ovde treba —
bez klase, bez `#` polja, bez konvencije `_ime`.

**Nepromenljiva vrednost.** Transakcija posle nastanka nema svoj životni tok: ne menja se, samo se **zamenjuje**
novom. Zbog toga se ista transakcija može bezbedno **deliti** između više verzija knjige.

**Validacija na granici.** Nedozvoljena vrednost se odbija pri konstrukciji, pa nijedna kasnija funkcija ne mora
da se brani od nepostojećeg statusa. Skup dozvoljenih vrednosti je podatak (`STATUSI`, `TIPOVI`), a ne niz `if`-ova.

### Mehanizmi

**`(() => { … })()`** — odmah pozvana funkcija. Napravi privatni prostor koji traje koliko i fabrika.

**`sledeciRedniBroj++`** — jedino mesto u celom rešenju gde se nešto menja.

**`Object.freeze`** — zabranjuje dodavanje, brisanje i izmenu polja. Zadatak traži da `opoziv` ne sme da menja
nijednu transakciju; zamrzavanje to garantuje **na nivou jezika**, a ne na nivou discipline pisanja koda.

**Skraćeni zapis polja** (`brojRacuna,` umesto `brojRacuna: brojRacuna`) — objekat se gradi jednim izrazom.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| privatni statički | promenljiva u zatvorenju — ne postoji sintaksa da joj se priđe | `static #brojac` unutar klase |
| nepromenljivost | `Object.freeze`, deljenje bez kopiranja | setteri i odbrambeno kopiranje pri svakom prosleđivanju |
| tip | običan objekat, „patka" tipizacija | `instanceof Transakcija`, jasna identifikacija tipa |
| validacija | u fabrici, pre nastanka objekta | u konstruktoru, često i u setterima |

**Prednosti.** Nema `this` ni `new`, pa nema greške sa izgubljenim kontekstom. Nepromenljiv objekat se bezbedno
deli između verzija. Konstrukcija i validacija su na jednom mestu.

**Mane.** Brojač je skriveno stanje: fabrika **nije čista funkcija** i ne može se resetovati u testu — u OOP bi
statičko polje bar bilo dostupno test kodu. Gubi se `instanceof` i, sa njim, deo pomoći editora.

---
# Jedinica 2 · Čiste funkcije nad nizom transakcija

In [ ]:
const sa  = (niz, transakcija) => [...niz, transakcija];                       // dodavanje — nov niz
const bez = (niz, redniBroj)   => niz.filter(x => x.redniBroj !== redniBroj);  // uklanjanje — nov niz

const stornirana = t => Object.freeze({ ...t, status: "stornirana" });         // kopija sa izmenjenim poljem

// razlika suma REALIZOVANIH: u korist minus na teret — u JEDNOM prolazu
const stanjeOd = niz => niz.reduce((s, x) =>
    x.status !== "realizovana" ? s :
    x.tip    === "u korist"    ? s + x.iznos
                               : s - x.iznos, 0);

### Koncepti

**Čista funkcija.** Rezultat zavisi samo od argumenata, i poziv ne ostavlja trag u svetu. Sve četiri funkcije
ovde su takve: `sa` ne dopisuje u prosleđeni niz nego vraća nov, `stornirana` ne menja status prosleđene
transakcije nego vraća njenu kopiju.

**Kopija sa izmenom umesto izmene.** `{ ...t, status: "stornirana" }` je funkcionalni odgovor na „promeni polje":
napravi se nov objekat koji je isti u svemu osim u jednom polju. Original ostaje netaknut, što je tačno ono
što `opoziv` mora da poštuje.

**Preklapanje (fold).** `stanjeOd` svodi niz na jedan broj. Umesto `for` petlje sa promenljivom `zbir` koja se
usput menja, opisuje se **kako se dva susedna koraka spajaju** — a JS pravi obilazak.

**Jedan prolaz.** Zbir bi se mogao napisati i kao `filter(...).filter(...).reduce(...)`, ali bi to bila tri
obilaska i dva međuniza. Ovde je uslov ugrađen u sam redjuser, pa je prolaz jedan i međurezultata nema.

### Mehanizmi

**`[...niz, x]`** — spread pravi nov niz; `push` bi menjao postojeći.

**`filter`** — vraća nov niz; nema uklanjanja na mestu (`splice`).

**`{ ...t, polje: novo }`** — plitka kopija sa izmenjenim poljem. Plitka je dovoljna jer su sva polja transakcije
proste vrednosti.

**Ugnježđeni ternarni izraz** — poravnat po kolonama, čita se kao tabela odluka. Svaka grana **vraća** vrednost,
pa nema `if` naredbi ni privremenih promenljivih.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| dodavanje u niz | `[...niz, x]` — nov niz, O(n) | `niz.push(x)` — isti niz, O(1) |
| izmena statusa | nova transakcija sa novim statusom | `t.setStatus("stornirana")` na istoj instanci |
| zbir | `reduce` — opis spajanja | `for` petlja sa akumulatorom koji se menja |
| pripadnost logike | slobodna funkcija nad podacima | metoda `Transakcija.prototype` / `Knjiga.izracunajStanje()` |

**Prednosti.** Funkcija koja ništa ne menja može se pozvati bilo kada i bilo koliko puta — otud i mogućnost da
sve provere stoje na kraju fajla. Nema pitanja „ko još drži referencu na ovaj niz".

**Mane.** Kopiranje niza pri svakom dodavanju je O(n) umesto O(1). Za velike nizove to se rešava trajnim
strukturama (deljena stabla), što je posao koji OOP verzija sa `push` nema.

---
# Digresija · Šta se stvarno dešava kad funkcija „pamti" promenljivu

Zašto su `sa`, `bez`, `stornirana` i `stanjeOd` napisane **van** tipa `Transakcija`, a `dodajTransakciju`
i `ukloniTransakciju` **unutar** `GlavnaKnjiga`? To nije stvar ukusa. Postoji pravilo koje odlučuje,
ali da bi imalo smisla, prvo treba videti šta se ispod zaista dešava.

## Tri činjenice iz kojih sledi sve ostalo

**1 · Svaki poziv pravi svoj zapis promenljivih.**
Kada JavaScript pozove funkciju, ne izvršava samo njeno telo. Prvo napravi zaseban zapis u memoriji —
**okruženje tog poziva** — u koji smešta parametre i sve što je unutra deklarisano sa `let`, `const` ili `var`.
Ne jedan zapis po funkciji, nego **jedan po pozivu**: deset poziva iste funkcije napravi deset odvojenih
zapisa, svaki sa svojim primerkom istih imena.

**2 · Vrednost funkcije nije samo kôd.**
Kada se u toku izvršavanja naiđe na definiciju funkcije, nastaje vrednost sastavljena od **dva dela**:
kôda i **pokazivača na zapis u kome je ta definicija izvršena**. Pokazivač se postavlja u trenutku
nastanka funkcije i posle se nikada ne menja. Funkcija je, dakle, par — posao plus adresa mesta na kome
stoje promenljive koje taj posao pominje.

**3 · Traženje imena ide lancem, a lanac određuje mesto pisanja.**
Kad telo funkcije pomene neko ime, ono se prvo traži u zapisu tekućeg poziva. Ako ga tamo nema, prelazi se
na zapis na koji ta funkcija pokazuje, pa na zapis na koji pokazuje **on**, i tako do najvišeg nivoa.
Presudno: taj lanac zavisi od toga **gde je kôd napisan**, a nimalo od toga ko funkciju poziva.

## Šta iz toga sledi

**Zapis preživljava povratak.** Sakupljač smeća briše zapis tek kad na njega niko više ne pokazuje.
Ako je funkcija koja na njega pokazuje vraćena napolje i negde sačuvana, zapis ostaje živ — iako je poziv
koji ga je napravio odavno završen. Promenljiva nastavlja da postoji, a jedino što je drži je ta funkcija.

**Deli se mesto, ne kopija.** Dve funkcije napravljene u istom pozivu dobile su **isti** pokazivač,
pa gledaju u isti zapis i u isto skladišno mesto u njemu. Upis kroz jednu odmah se vidi kroz drugu.
Nije snimljena vrednost u trenutku nastanka — to je bukvalno ista promenljiva.

**Odvojeni pozivi ne znaju jedan za drugi.** Različiti pozivi → različiti zapisi → različita mesta,
iako se ime piše isto.

**Spolja ne postoji izraz kojim bi se do tog mesta došlo.** Zapis nije objekat: nije ničije polje,
ne pojavljuje se u `Object.keys`, ne dohvata se tačkom. Jedini pristup je kroz funkcije koje na njega
pokazuju, i to samo onim imenima koja te funkcije pominju u svom kôdu. Privatnost ovde ne dolazi od
ključne reči, nego od **nepostojanja puta**.

Primer ispod pokazuje svih pet tvrdnji redom.

In [ ]:
// ═══ 1 · zapis poziva preživljava povratak iz funkcije ════════════════

const napraviCitac = () => {
    const tajna = "vrednost iz poziva koji je odavno završen";
    return () => tajna;                  // ova funkcija nosi pokazivač na zapis
};

const citac = napraviCitac();            // napraviCitac je ZAVRŠIO
console.log("1) posle povratka i dalje čita:", citac());

// ═══ 2 · traži se po MESTU PISANJA, ne po mestu poziva ════════════════

const poruka = "napisana na najvišem nivou";

const citajPoruku = () => poruka;        // napisana ovde → gleda u ovaj zapis

const drugiPozivalac = () => {
    const poruka = "napisana unutar drugog poziva";
    return [citajPoruku(), poruka];      // poziva se ODAVDE, ali to ništa ne menja
};

console.log("2) poziv iz drugog okruženja:", drugiPozivalac());

// ═══ 3 · dve funkcije iz ISTOG poziva dele isto skladišno mesto ═══════

const napraviPar = (pocetna) => {
    let vrednost = pocetna;              // jedno mesto u ovom zapisu
    return {
        upisi:    x => { vrednost = x; },
        procitaj: () => vrednost
    };
};

const par1 = napraviPar(0);
console.log("3) pre upisa:", par1.procitaj());
par1.upisi(42);
console.log("   posle upisa kroz drugu funkciju:", par1.procitaj(), "— isto mesto, nije kopija");

// ═══ 4 · dva poziva fabrike = dva odvojena zapisa ═════════════════════

const par2 = napraviPar(0);
par1.upisi(100);
par2.upisi(200);
console.log("4) par1:", par1.procitaj(), "| par2:", par2.procitaj(), "— nezavisni");

// ═══ 5 · spolja nema izraza kojim bi se došlo do tog mesta ════════════

console.log("5) polja objekta:      ", Object.keys(par1));
console.log("   par1.vrednost:      ", par1.vrednost);
console.log("   ime vrednost ovde:  ", typeof vrednost);

In [ ]:
// ═══ 6 · koliko zapisa pravi petlja ═══════════════════════════════════
// Ista petlja, ista tri poziva, dva različita ishoda — razlika je isključivo
// u tome koliko je zapisa napravljeno.

const uPetlji = (() => {
    const saVar = [];
    for (var i = 0; i < 3; i++) saVar.push(() => i);    // JEDAN zapis za celu petlju

    const saLet = [];
    for (let j = 0; j < 3; j++) saLet.push(() => j);    // NOV zapis pri svakom prolazu

    return { saVar: saVar.map(f => f()), saLet: saLet.map(f => f()) };
})();

console.log("6) tri funkcije napravljene uz var:", uPetlji.saVar);
console.log("   tri funkcije napravljene uz let:", uPetlji.saLet);

### Čitanje ispisa

**Red 1.** `napraviCitac` je vratio vrednost i završio, ali `tajna` i dalje postoji. Nju drži jedino
vraćena funkcija. Da je nismo sačuvali u `citac`, zapis bi bio obrisan.

**Red 2.** Ovo je najvažniji red. `citajPoruku` je *napisana* na najvišem nivou, a *pozvana* iz
`drugiPozivalac`, gde postoji drugo ime `poruka`. Vraća `"napisana na najvišem nivou"` — dakle gleda u
zapis svog **mesta pisanja**, ne pozivaoca. Da JavaScript gleda pozivaoca, oba stringa bila bi ista.

**Red 3.** `upisi` i `procitaj` nastale su u istom pozivu `napraviPar`. Upis kroz prvu pročita se kroz drugu.
Da je svaka dobila kopiju, `procitaj()` bi i dalje vraćao `0`.

**Red 4.** `par2` je drugi poziv iste fabrike, pa ima svoj zapis. `100` i `200` se ne mešaju.

**Red 5.** Objekat ima samo `upisi` i `procitaj`. `vrednost` nije polje (`undefined`) i ne postoji kao ime
u ovom dosegu (`typeof` daje `"undefined"` umesto da baci grešku). Nema načina da joj se priđe.

**Red 6.** `var i` pravi jedno mesto za celu funkciju, pa sve tri napravljene funkcije pokazuju na isti
zapis i vide poslednju vrednost — `[3, 3, 3]`. `let j` pravi nov zapis pri svakom prolazu, pa svaka funkcija
ima svoj primerak — `[0, 1, 2]`. Isti kôd, a ishod zavisi samo od broja napravljenih zapisa.

### Zašto je ovo mehanizam privatnosti

Zato što je jedini put do zapisa — funkcija koja na njega pokazuje, i to samo kroz imena koja ta funkcija
pominje. `procitaj` vraća `vrednost` jer je tako napisana; da nije, tog imena ne bi bilo odakle dohvatiti.
Nema deklaracije koja bi pristup zabranila, nema modifikatora — jednostavno **ne postoji izraz** koji vodi tamo.

*(Ovaj mehanizam se u ostatku notebooka — Jedinica 1 i Jedinica 3 — pominje jednom rečju.
Ovde je namerno opisan samo kroz ono što se stvarno dešava.)*

---

## Pravilo gde funkcija sme da stoji

Iz treće činjenice — da ime mora postojati u lancu zapisa mesta na kome je funkcija **napisana** — sledi pravilo:

> Pogledaj koja imena funkcija pominje, a koja **nisu njeni parametri**.
> Ako pominje ime koje postoji samo u zapisu nekog poziva → **mora biti napisana unutar tog poziva**.
> Ako joj trebaju samo njeni argumenti → **ide napolje**.

Primer ispod pokazuje oba slučaja na generičkom kodu, nezavisno od glavne knjige.

In [ ]:
// ═══ SLUČAJ A · funkciji treba nešto privatno → MORA unutra ═══════════

const Sekvenca = (pocetak = 1) => {
    let n = pocetak;                  // privatno: postoji samo unutar OVOG poziva fabrike

    return Object.freeze({
        sledeci: () => n++,           // pominje n — a n nije parametar → dolazi iz zapisa poziva
        dokle:   () => n              // pominje ISTI n
    });
};

const sekvencaA = Sekvenca(1);
const sekvencaB = Sekvenca(100);

console.log("1) svaki poziv fabrike ima svoje n:",
            sekvencaA.sledeci(), sekvencaA.sledeci(), "| druga sekvenca:", sekvencaB.sledeci());

console.log("2) dve metode iste instance dele isto n:", sekvencaA.dokle());   // 3, ne 1

console.log("3) n nije polje objekta:", sekvencaA.n,
            "| n ne postoji ni kao ime spolja:", typeof n);

// pokušaj da se ista operacija napiše NAPOLJU — nema šta da uveća
const sledeciSpolja = sekv => sekv.n++;

try { console.log("spolja:", sledeciSpolja(sekvencaA)); }
catch (g) { console.log("4) spolja ne može:", g.constructor.name, "— nema polja n, a objekat je zamrznut"); }

In [ ]:
// ═══ SLUČAJ B · funkciji trebaju samo argumenti → ide NAPOLJE ═════════

const Kutija = (sadrzaj = []) => {
    const s = Object.freeze([...sadrzaj]);      // privatno

    return Object.freeze({
        get stavke() { return [...s]; },
        dodaj: x => Kutija([...s, x])           // pominje s → MORA unutra (isto kao dodajTransakciju)
    });
};

// pominju samo svoje parametre → nemaju razloga da budu unutra (isto kao stanjeOd)
const najveci = niz => niz.reduce((m, x) => x > m ? x : m, -Infinity);
const duzina  = niz => niz.length;

const kutija1 = Kutija([3, 9, 4]);
const kutija2 = kutija1.dodaj(12);

console.log("1) dodaj vraća novu kutiju:", kutija1.stavke, "→", kutija2.stavke);

// šta se dobija time što su napolju:
console.log("2) rade nad kutijom:      ", najveci(kutija2.stavke), duzina(kutija2.stavke));
console.log("3) rade nad PODSKUPOM:    ", najveci(kutija2.stavke.filter(x => x < 10)));
console.log("4) rade i bez kutije:     ", najveci([7, 2, 5]));
console.log("5) prosleđuju se kao vrednost:", [[1, 5], [9, 2], [3]].map(najveci));

// cena stavljanja unutra: fabrika pravi NOVU funkciju za svaku instancu
console.log("6) dve kutije dele metodu dodaj?", Kutija([]).dodaj === Kutija([]).dodaj);

### Čitanje ispisa

**Slučaj A.** Red 1 pokazuje da su `sekvencaA` i `sekvencaB` potpuno nezavisne — svaki poziv fabrike napravio je
svoj `n`. Red 2 pokazuje suprotnu stranu iste stvari: `sledeci` i `dokle` nastali su u **istom** pozivu, pa dele
isto `n` — `dokle()` vraća 3, a ne početnu vrednost. Red 3 pokazuje da `n` nije polje objekta i da spolja ne
postoji ni kao ime. Red 4 je pokušaj da se `sledeci` napiše napolju: nema šta da uveća, a upis u zamrznut
objekat baca grešku. **Zato `sledeci` nema izbora — mora biti napisan unutar fabrike.**

**Slučaj B.** `dodaj` pominje `s`, pa je i on unutra iz istog razloga. Ali `najveci` i `duzina` pominju samo
svoje parametre, i redovi 3–5 pokazuju šta se time dobija: rade nad podskupom, nad nizom koji nije ničija kutija,
i prosleđuju se kao vrednost u `map`. Metoda ne bi mogla nijedno od toga bez omotača.
Red 6 je cena: fabrika bez prototipa pravi **novu** funkciju za svaku instancu, pa `===` daje `false`.

### Primenjeno na ovaj zadatak

| Funkcija | Pominje li nešto van svojih parametara | Gde stoji |
| --- | --- | --- |
| `dodajTransakciju` | da — privatni niz `t` i referencu `knjiga` | unutar `GlavnaKnjiga` |
| `ukloniTransakciju` | da — isto | unutar `GlavnaKnjiga` |
| `sa`, `bez` | ne — samo `niz` i drugi argument | napolje |
| `stanjeOd` | ne — samo `niz` | napolje |
| `stornirana` | ne — samo `t` | napolje |

Kod `sa`, `bez` i `stanjeOd` postoji i drugi razlog: primaju **niz** transakcija, ne jednu. Nijedna pojedinačna
transakcija ne bi mogla biti vlasnik operacije nad svim ostalima. A pošto su napolju, `stanjeOd` može da radi
i nad podskupom — `stanjeOd(pretraga([uMesecu("2026-08")], k))` daje stanje samo za avgust,
što geter `k.stanje` ne može jer uvek računa nad celim nizom.

### Zašto baš `stornirana` ne sme da bude metoda

Ona je jedina koja prima **jednu** transakciju, pa deluje kao prirodan kandidat za metodu `t.storniraj()`.
Ipak ne valja, iz dva razloga:

**1. Redni broj.** Metoda koja pravi novu transakciju morala bi da pozove fabriku, a fabrika **uvećava brojač** —
stornirana kopija dobila bi nov redni broj. A stornirana transakcija je *ista* transakcija sa drugim statusom
i mora zadržati svoj redni broj. Sa slobodnom funkcijom i spread-om (`{ ...t, status }`) redni broj se prenosi.

**2. Prosleđivanje.** `opoziv` se završava sa `.map(stornirana)` — funkcija se prosleđuje kao vrednost.
Sa metodom bi to bilo `.map(x => x.storniraj())`, a direktno prosleđivanje metode ne bi puklo nego bi **tiho**
dalo pogrešan rezultat: metoda pamti svoju transakciju iz zapisa u kome je nastala i ignoriše ono što joj `map` šalje.

### Šta se ovim gubi

Pravilo nije besplatno. Slobodna funkcija nema `instanceof` proveru ni ograničenje nad čim se poziva —
`stanjeOd([1, 2, 3])` se ne buni dok ne pukne na `x.status`. U objektnom pristupu bi tip to sprečio unapred.
Takođe, `k.stanje` se čita bez uvoza, dok `stanjeOd(niz)` traži da se zna gde ta funkcija stoji.
Rešenje je oba zadržalo: `stanje` kao geter za uobičajen slučaj, `stanjeOd` kao slobodnu funkciju za sve ostale.

---
# Jedinica 3 · `GlavnaKnjiga`, izvedeno stanje i veza sa prethodnom verzijom

In [ ]:
const GlavnaKnjiga = (naziv, maticniBroj, pib, transakcije = [], prethodna = null) => {
    const t = Object.freeze([...transakcije]);   // privatna zamrznuta kopija — spolja se ne dohvata

    const knjiga = Object.freeze(Object.defineProperties({
        naziv, maticniBroj, pib,
        prethodna,          // celokupna istorija izmena: veza sa knjigom od koje je ova nastala

        // obe metode vraćaju NOVU knjigu; nad ovom se ne menja ništa
        dodajTransakciju:  tr        => GlavnaKnjiga(naziv, maticniBroj, pib, sa(t, tr),         knjiga),
        ukloniTransakciju: redniBroj => GlavnaKnjiga(naziv, maticniBroj, pib, bez(t, redniBroj), knjiga)
    }, {
        transakcije: { get: () => [...t],      enumerable: true },   // uvek kopija ka spolja
        stanje:      { get: () => stanjeOd(t), enumerable: true }    // izvedeni atribut
    }));

    return knjiga;
};

### Koncepti

**Trajna (persistent) struktura podataka.** Izmena ne uništava staru vrednost — pravi novu, a stara i dalje važi.
Pošto se transakcije **dele** (nisu kopirane, samo su u novom nizu), nova verzija košta koliko i kopiranje niza
pokazivača. Istorija time nije nešto što se dodatno vodi, nego **posledica načina gradnje**.

**Izvedeni atribut.** `stanje` se ne čuva nego se računa pri svakom čitanju. Zbog toga ne može da se „raziđe"
sa sadržajem knjige — nema dve vrednosti koje treba držati usklađenim. Cena je ponovno računanje pri svakom pristupu.

**Enkapsulacija kroz zatvorenje.** Niz `t` nije polje objekta nego promenljiva u zatvorenju. Ne postoji `knjiga.t`.
Ono što spolja liči na atribut `transakcije` je geter koji vraća **kopiju**, pa ni `knjiga.transakcije.push(x)`
ne može da dopre do originala.

**Referenca `prethodna` = jednostruko povezana lista verzija.** Knjiga ne zna svoju budućnost, samo svoju prošlost.
Zato je lanac aciklički i staru verziju je nemoguće „pokvariti" novom.

### Mehanizmi

**Fabrička funkcija.** Objekat se pravi i vraća unutar funkcije. Nema `this`, pa se metoda može proslediti dalje
bez vezivanja: `verzije.map(k => k.stanje)` radi jer je `stanje` geter nad zatvorenjem, a ne nad `this`.

**Rekurzivni poziv `GlavnaKnjiga(...)` u metodama** — nova verzija nastaje istim putem kao i prva. Nema posebne
grane za „izmenu".

**Vezivanje `const knjiga = …` pa upotreba `knjiga` unutar metoda.** Metode moraju da proslede **ovu** knjigu kao
`prethodna`. Pošto se objekat gradi jednim izrazom, vrednost se prvo veže za ime, a strelice je vide iz zatvorenja —
pozivaju se tek kasnije, kada je vezivanje odavno gotovo.

**`Object.defineProperties` sa `get`** — `stanje` i `transakcije` se čitaju **bez zagrada**, kao obična polja,
iako iza njih stoji izračunavanje. `enumerable: true` ih vraća u `Object.keys` i `console.log`.

**`Object.freeze` na kraju** — sprečava da neko dopiše ili zameni metodu na gotovoj knjizi.

### Pretpostavka

Tekst kaže da knjiga sadrži „celokupnu istoriju izmena u vidu reference na prethodnu glavnu knjigu".
Uzeto je doslovno: čuva se **jedna** referenca (`prethodna`), a celokupna istorija se iz nje dobija razvijanjem
lanca (jedinica 4). Alternativa — čuvati ceo niz verzija u svakoj knjizi — davala bi kvadratnu potrošnju memorije.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| izmena | nova instanca, stara ostaje važeća | `this.transakcije.push(...)`, ista instanca |
| istorija | posledica gradnje (`prethodna`) | poseban `Memento` / `UndoStack` koji se održava ručno |
| `stanje` | geter koji računa iz izvora | polje koje se ažurira u svakom setteru |
| privatnost | promenljiva u zatvorenju | `private` / `#` polje |
| identitet | svaka verzija je nov objekat | jedan objekat kroz ceo život, `===` stabilan |
| cena dodavanja | O(n) — kopira se niz | O(1) — `push` |

**Prednosti.** Ko drži `knjiga1` siguran je da mu se sadržaj neće promeniti pod rukama — nema greške deljene
reference. Svaka verzija je konzistentan snimak, pogodan za poređenje i poništavanje. „Undo" i revizioni trag
dobijaju se besplatno. Nema para vrednosti (`transakcije`, `stanje`) koje bi mogle da se raziđu.

**Mane.** Svaka izmena troši memoriju za nov objekat i nov niz, a lanac verzija raste neograničeno — u dugotrajnoj
aplikaciji mora se rezati. Identitet objekta se menja pri svakoj izmeni, pa se „ista knjiga" prati kroz promenljivu,
a ne kroz referencu. `stanje` se preračunava pri svakom čitanju.

---
# Jedinica 4 · `istorija` — lanac verzija razvijen u niz

**Pretpostavka.** „Redni broj glavne knjige u istoriji" tumači se kao **indeks u nizu verzija**, gde je `0` prva
(početna) knjiga, a poslednji indeks trenutna. Funkcija `istorija` upravo taj niz i vraća.

In [ ]:
const istorija = knjiga => knjiga === null ? [] : [...istorija(knjiga.prethodna), knjiga];

### Koncepti

**Rekurzija kao definicija, ne kao postupak.** Red se čita: *istorija ničega je prazan niz; istorija knjige je
istorija njene prethodnice, pa zatim ona sama.* Nema brojača, nema `let`, nema niza koji se usput menja —
opisano je **šta istorija jeste**, a ne kako je sastaviti.

**Osnovni slučaj i korak.** `null` je kraj lanca (prva knjiga nema prethodnicu). Svaki korak skida jedan član,
pa se rekurzija sigurno završava — lanac je konačan i aciklički.

**Redosled je posledica mesta rekurzivnog poziva.** Pošto `istorija(knjiga.prethodna)` stoji **ispred** `knjiga`,
niz izlazi poređan od najstarije ka najnovijoj. Da je obrnuto, dobio bi se obrnut redosled — bez ijedne druge izmene.

### Mehanizmi

**Ternarni izraz** — cela funkcija je jedan izraz koji vraća vrednost, pa može biti strelica bez tela.

**Spread u literalu niza** (`[...istorija(prethodna), knjiga]`) — spaja rezultat rekurzije i tekući član u nov niz.

**Bez repne optimizacije.** JS ne garantuje repnu rekurziju, a ovaj poziv ionako nije repni (spread se dešava posle
povratka). Za lanac od nekoliko hiljada verzija to bi bilo pitanje; za kolokvijumski obim nije.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| obilazak lanca | rekurzija, jedan izraz | `while (k !== null) { niz.unshift(k); k = k.prethodna; }` |
| rezultat | nov niz pri svakom pozivu | često se čuva i održava kao polje |
| redosled | određen mestom rekurzivnog poziva | određen izborom `push` ili `unshift` |

**Prednosti.** Nema promenljive koja se menja u petlji, pa nema ni klase grešaka vezanih za nju. Funkcija je čista:
za istu knjigu uvek isti niz, pa se sme pozvati i u proveri i u `opoziv`-u bez razmišljanja o redosledu.

**Mane.** Pri svakom pozivu se gradi nov niz — `opoziv` je zato poziva jednom i rezultat drži u promenljivoj.
Dubina rekurzije je jednaka broju verzija.

---
# Jedinica 5 · `pretraga` — kriterijumi kao funkcije

In [ ]:
// preklapanje niza kriterijuma: izlaz jednog filtera je ulaz sledećeg,
// pa je redosled iz argumenata ujedno i redosled primene.
const pretraga = (kriterijumi, knjiga) =>
    kriterijumi.reduce((niz, kriterijum) => niz.filter(kriterijum), knjiga.transakcije);

// kriterijumi su obične predikatske funkcije — pišu se nezavisno od pretrage
const realizovane   = t => t.status === "realizovana";
const nerealizovane = t => t.status === "nerealizovana";
const stornirane    = t => t.status === "stornirana";
const uKorist       = t => t.tip === "u korist";
const naTeret       = t => t.tip === "na teret";
const preko         = granica => t => t.iznos > granica;               // parametrizovan kriterijum
const uMesecu       = mesec   => t => t.datum.slice(0, 7) === mesec;
const saRacuna      = broj    => t => t.brojRacuna === broj;

### Koncepti

**Funkcija višeg reda.** `pretraga` prima funkcije kao podatke i ne zna ništa o tome šta one proveravaju.
Novi kriterijum se dodaje bez ijedne izmene u `pretraga` — pretraga je otvorena za proširenje, a zatvorena za izmenu.

**Predikat.** Funkcija koja vraća `true`/`false` za jedan element. Svih osam kriterijuma su predikati, pa su
međusobno zamenljivi i mogu se slagati u bilo kom broju i redosledu.

**Funkcija koja vraća funkciju (currying / parcijalna primena).** `preko` nije predikat — to je **fabrika predikata**.
`preko(50000)` tek daje predikat, i taj predikat pamti granicu u zatvorenju. Isti obrazac je i kod `uMesecu` i `saRacuna`.

**Preklapanje nad funkcijama.** `reduce` ovde ne sabira brojeve nego **lančano sužava niz**: početna vrednost je
niz svih transakcija, a svaki korak ga propušta kroz jedan filter. Time redosled iz argumenata postaje redosled primene,
što zadatak izričito traži.

**Kriterijumi kao podatak koji se sastavlja u toku rada.** Pošto stižu kao **niz**, mogu se sastaviti od popunjenih
polja obrasca — što je pokazano u proverama. U verziji sa promenljivim brojem argumenata (`...kriterijumi`) to bi
tražilo `apply` ili dodatni spread.

### Mehanizmi

**`reduce` sa nizom kao akumulatorom** — akumulator ne mora biti broj; ovde je to niz transakcija.

**`filter` u svakom koraku** — vraća nov niz, pa se ništa ne menja na mestu.

**Zatvorenje u `preko(granica)`** — vraćena strelica pamti `granica` i posle završetka spoljnog poziva.

**Prazan niz kriterijuma** — `reduce` sa početnom vrednošću vraća baš tu početnu vrednost, pa `pretraga([], k)`
prirodno daje sve transakcije. Nema posebne grane za taj slučaj.

**Redosled primene je merljiv.** Isti skup rezultata dobija se za `[A, B]` i `[B, A]`, ali **broj poziva** svakog
predikata je različit: prvi navedeni vidi sve transakcije, drugi samo one koje su prošle prvi. To je i provereno u DEO II.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| kriterijum | obična funkcija | objekat sa metodom `Specification.isSatisfiedBy(t)` |
| kombinovanje | niz funkcija + `reduce` | `AndSpecification(a, b)`, kompozitni obrazac |
| parametrizacija | `preko(50000)` vraća funkciju | `new IznosPreko(50000)` — instanca sa poljem |
| proširenje | dopiše se jedna strelica | nova klasa koja implementira interfejs |

**Prednosti.** Kriterijum je jedan red koda, bez klase i bez interfejsa. Kombinovanje je obično nizanje.
Pošto su predikati čiste funkcije, pretraga je ponovljiva i lako se testira.

**Mane.** Funkcija nema ime u vreme izvršavanja pa je teže napraviti čitljiv opis pretrage ili je serijalizovati
(npr. pretvoriti u SQL upit ili sačuvati u bazu) — objektni `Specification` se lako obiđe i prevede.
Za velike nizove, `n` uzastopnih `filter` poziva pravi `n` međunizova; transdjuseri sa Kolokvijuma II
rešavaju baš taj problem u jednom prolazu.

---
# Jedinica 6 · `opoziv` — nova knjiga iz zadate verzije

**Pretpostavka.** „Transakcije nastale nakon te glavne knjige" su one koje se pojavljuju u verzijama posle zadatog
indeksa, a nisu postojale u toj verziji — prepoznaju se po **rednom broju**. Nova knjiga nastaje od **trenutne**
verzije, pa je njena `prethodna` upravo prosleđena knjiga i istorija ostaje neprekinuta.

In [ ]:
const opoziv = (knjiga, redniBrojUIstoriji) => {
    const verzije = istorija(knjiga);
    const stara   = verzije[redniBrojUIstoriji];
    if (!stara) throw new Error(`Ne postoji verzija sa indeksom ${redniBrojUIstoriji}`);

    const bileRanije = new Set(stara.transakcije.map(x => x.redniBroj));

    // "nastale nakon te glavne knjige" = sve što se javlja u kasnijim verzijama,
    // a nije bilo u zadatoj verziji; svaka se prepisuje u storniranu KOPIJU.
    const nastaleKasnije = verzije
        .slice(redniBrojUIstoriji + 1)
        .flatMap(k => k.transakcije)
        .filter(x => !bileRanije.has(x.redniBroj))
        .filter((x, i, niz) => niz.findIndex(y => y.redniBroj === x.redniBroj) === i)   // bez ponavljanja
        .map(stornirana);                                                               // originali ostaju netaknuti

    // nova knjiga se nastavlja na knjigu nad kojom je opoziv pozvan — istorija se ne prekida
    return GlavnaKnjiga(knjiga.naziv, knjiga.maticniBroj, knjiga.pib,
                        [...stara.transakcije, ...nastaleKasnije], knjiga);
};

### Koncepti

**Cevovod (pipeline).** Obrada je niz imenovanih koraka, svaki sa jednim poslom: uzmi kasnije verzije → skupi
njihove transakcije → izbaci one koje su već postojale → izbaci ponavljanja → storniraj. Svaki korak vraća nov niz,
pa se međurezultat može ispisati na bilo kom mestu bez uticaja na ostatak.

**Kopija umesto izmene, drugi put.** `.map(stornirana)` proizvodi **nove** transakcije. Zahtev „ne sme da menja
nijednu od glavnih knjiga niti neku od njihovih transakcija" ovde nije stvar pažnje pri pisanju — ispunjen je
zato što nijedna korišćena operacija ne ume da menja.

**Razlika skupova.** „Nastale kasnije" je skupovna razlika: sve iz kasnijih verzija minus sve iz zadate.
`Set` rednih brojeva pretvara proveru pripadnosti iz O(n) u O(1).

**Nastavljanje istorije umesto grananja.** Opoziv ne briše verzije koje poništava — dodaje **novu** verziju
na vrh lanca. Time i sam opoziv postaje događaj u istoriji, koji se može opozvati.

### Mehanizmi

**`istorija(knjiga)` pozvan jednom** — funkcija je čista, ali gradi nov niz, pa se rezultat vezuje za promenljivu.

**`slice(i + 1)`** — samo verzije posle zadate; `slice` vraća nov niz, ne menja postojeći.

**`flatMap`** — spaja niz nizova u jedan niz; zamenjuje ugnježđenu petlju.

**`filter` sa `findIndex` za jedinstvenost** — zadržava se samo prvo pojavljivanje svakog rednog broja.
`Set` ovde ne bi pomogao jer se porede **objekti**, a isti redni broj mogu nositi različite reference.

**`map(stornirana)` — funkcija kao vrednost.** Prosleđuje se ime funkcije, bez omotača `x => stornirana(x)`.

**`throw` na nepostojeći indeks** — `verzije[99]` je `undefined`, pa bi bez provere greška izbila kasnije i
na nejasnom mestu.

### Zašto stanje opozvane knjige mora biti jednako stanju stare verzije

Sve što je nastalo kasnije dobija status `"stornirana"`, a `stanjeOd` sabira **samo realizovane**. Ostaju,
dakle, tačno one transakcije koje je zadata verzija imala, sa nepromenjenim statusima. Ta jednakost je u DEO II
uzeta kao provera — nije posebno programirana, nego je posledica definicije.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| poništavanje | nova verzija iz starog snimka | `undo()` koji vraća stanje iz `Memento` steka |
| „ne menjaj original" | garantovano tipom operacija | obaveza programera; lako se prekrši `t.status = …` |
| istorija posle opoziva | nastavlja se, opoziv je novi član | često se odseca — grana se gubi |
| međukoraci | imenovani, svaki vraća nov niz | promenljive koje se menjaju u petlji |

**Prednosti.** Zahtev o nemutiranju je ispunjen po konstrukciji, a ne po disciplini. Opoziv opoziva je moguć.
Svaki korak cevovoda se može zasebno proveriti.

**Mane.** Pet uzastopnih prolaza kroz niz umesto jednog. Provera jedinstvenosti sa `findIndex` je O(n²) —
za veliki broj transakcija tražila bi `Map` po rednom broju. Lanac verzija posle više opoziva raste brzo.

---
# DEO II — PROVERE

Sve što se izvršava i ispisuje stoji ovde, na kraju. Provera nije goli `console.log` nego poređenje
**dobijenog** i **očekivanog**, pa se odmah vidi da li nešto ne valja; na kraju ide izveštaj `N/N provera prošlo`.

**Zašto je ovakav raspored uopšte moguć.** Zato što su sve funkcije iz DEO I čiste. Kod sa promenljivim stanjem
morao bi da se proverava na tačnom mestu u toku programa, jer bi svaki kasniji poziv menjao ono što se proverava.
Ovde se međurezultati mogu držati u promenljivim koliko god dugo i porediti bilo kada — vrednost se ne kvari.

In [ ]:
// ── alat za provere ───────────────────────────────────────────────────
let ukupno = 0, prosle = 0;

const isti = (a, b) => JSON.stringify(a) === JSON.stringify(b);

const proveri = (opis, dobijeno, ocekivano) => {
    ukupno++;
    const ok = isti(dobijeno, ocekivano);
    if (ok) prosle++;
    console.log(`${ok ? "✔" : "✘"} ${opis}\n     dobijeno:  ${JSON.stringify(dobijeno)}` +
                (ok ? "" : `\n     očekivano: ${JSON.stringify(ocekivano)}`));
};

const greskaOd = f => { try { f(); return null; } catch (g) { return g.message; } };
const naslov   = tekst => console.log(`\n═══ ${tekst} ═══`);

### Koncepti

**Provera kao poređenje vrednosti.** Pošto su funkcije čiste, dovoljno je uporediti povratnu vrednost sa očekivanom —
nema stanja koje bi trebalo prvo namestiti niti počistiti posle. Otud i nema `setUp`/`tearDown`.

**Poređenje po strukturi.** `JSON.stringify` poredi **sadržaj**, a ne referencu. To je ovde tačno ono što treba:
`[1, 2] === [1, 2]` je `false`, ali dve različite reference sa istim sadržajem jesu isti rezultat.
Ograničenje: ne razlikuje `undefined` od izostalog polja i ne barata cikličnim strukturama — za ovaj zadatak nijedno ne smeta.

**Greška kao vrednost.** `greskaOd` pretvara `throw` u običnu povratnu vrednost (`null` ili poruka), pa provera
neuspeha izgleda isto kao i svaka druga provera. To je isti obrazac koji bi se dobio tipom „uspeh ili greška".

**Efekat na ivici.** `proveri` je jedina funkcija u celom rešenju koja ispisuje i menja brojače. Sve nečisto
skupljeno je na jedno mesto, van računanja.

In [ ]:
// ── nekoliko instanci tipa GlavnaKnjiga ───────────────────────────────
const t1 = Transakcija("265-0001", "Uplata kupca",   "realizovana",   "2026-08-01", "u korist", 120000);
const t2 = Transakcija("265-0001", "Zakup prostora", "realizovana",   "2026-08-03", "na teret",  45000);
const t3 = Transakcija("265-0001", "Struja",         "realizovana",   "2026-08-06", "na teret",  18000);
const t4 = Transakcija("265-0002", "Nabavka robe",   "nerealizovana", "2026-09-01", "na teret",  90000);

const k0 = GlavnaKnjiga("Merkur doo", "21456789", "108456789");   // verzija 0 — prazna
const k1 = k0.dodajTransakciju(t1);                               // verzija 1
const k2 = k1.dodajTransakciju(t2);                               // verzija 2
const k3 = k2.dodajTransakciju(t3);                               // verzija 3
const k4 = k3.dodajTransakciju(t4);                               // verzija 4 — tekuća

// druga knjiga, sagrađena preklapanjem — nezavisna od prve
const vega = ["Prodaja usluga|realizovana|u korist|2026-08-10|150000",
              "Plate|realizovana|na teret|2026-08-11|95000",
              "Reprezentacija|nerealizovana|na teret|2026-08-12|12000"]
    .map(red => red.split("|"))
    .reduce((k, [opis, status, tip, datum, iznos]) =>
                k.dodajTransakciju(Transakcija("170-9999", opis, status, datum, tip, Number(iznos))),
            GlavnaKnjiga("Vega ad", "20099887", "105566778"));

console.log("verzije prve knjige:", istorija(k4).length, "| druge:", istorija(vega).length);

### Mehanizmi

**Lanac se gradi imenovanim vezivanjima** (`k0` … `k4`). Svako ime je jedna verzija, pa se u proverama može
uporediti bilo koja sa bilo kojom.

**Druga knjiga se gradi preklapanjem** — `reduce` prolazi kroz opise i za svaki poziva `dodajTransakciju`,
pri čemu je akumulator **sama knjiga**. Isti mehanizam kao `stanjeOd`, samo je akumulator objekat umesto broja.
Time je i sam ulazni podatak (`"opis|status|tip|datum|iznos"`) odvojen od koda koji ga gradi.

**Razlaganje niza u parametrima** (`([opis, status, tip, datum, iznos]) => …`) — red iz `split("|")` se odmah
razlaže u imenovane delove.

In [ ]:
// ── 1 · Transakcija ───────────────────────────────────────────────────
naslov("1 · Transakcija");

proveri("redni broj se uvećava za jedan pri svakom instanciranju",
        [t2.redniBroj - t1.redniBroj, t3.redniBroj - t2.redniBroj, t4.redniBroj - t3.redniBroj], [1, 1, 1]);

proveri("svi traženi podaci postoje",
        Object.keys(t1), ["redniBroj", "brojRacuna", "opis", "status", "datum", "tip", "iznos"]);

proveri("nedozvoljen status se odbija",
        greskaOd(() => Transakcija("265-0001", "X", "izmisljen", "2026-08-04", "u korist", 1)) !== null, true);

proveri("nedozvoljen tip se odbija",
        greskaOd(() => Transakcija("265-0001", "X", "realizovana", "2026-08-04", "u dobit", 1)) !== null, true);

// NAMERNO: pokušaj izmene zamrznute transakcije — u strogom režimu baca, inače tiho ne uspeva
try { t1.iznos = 999999; } catch (g) {}
proveri("transakcija je zamrznuta — izmena ne prolazi", [t1.iznos, Object.isFrozen(t1)], [120000, true]);

proveri("brojač je privatan — spolja ga nema", typeof sledeciRedniBroj, "undefined");

// ── 2 · GlavnaKnjiga i izvedeni atribut stanje ────────────────────────
naslov("2 · GlavnaKnjiga i izvedeni atribut stanje");

proveri("naziv, matični broj i pib",
        [k4.naziv, k4.maticniBroj, k4.pib], ["Merkur doo", "21456789", "108456789"]);

proveri("transakcije su niz Transakcija objekata", k4.transakcije.length, 4);

proveri("stanje = realizovane u korist − realizovane na teret",
        k4.stanje, 120000 - 45000 - 18000);              // 57000; nerealizovana se ne računa

proveri("stanje je izvedeno, ne uskladišteno — prazna knjiga daje nulu", k0.stanje, 0);

k4.transakcije.push(t1);                                 // NAMERNO: geter vraća kopiju
proveri("geter transakcije vraća kopiju — knjiga ostaje ista", k4.transakcije.length, 4);

proveri("knjiga je zamrznuta", Object.isFrozen(k4), true);

proveri("druga instanca je nezavisna",
        [vega.naziv, vega.stanje, k4.naziv, k4.stanje], ["Vega ad", 55000, "Merkur doo", 57000]);

In [ ]:
// ── 3 · dodajTransakciju i ukloniTransakciju ne mutiraju original ─────
naslov("3 · dodajTransakciju i ukloniTransakciju");

const dodata    = k4.dodajTransakciju(Transakcija("265-0001", "Kamata", "realizovana", "2026-09-05", "u korist", 4000));
const uklonjena = k4.ukloniTransakciju(t2.redniBroj);

proveri("dodavanje daje novu knjigu sa transakcijom više",
        [dodata.transakcije.length, dodata.stanje], [5, 61000]);

proveri("uklanjanje daje novu knjigu bez zadate transakcije",
        [uklonjena.transakcije.map(x => x.opis), uklonjena.stanje],
        [["Uplata kupca", "Struja", "Nabavka robe"], 102000]);

proveri("original nije mutiran nijednom metodom",
        [k4.transakcije.length, k4.stanje], [4, 57000]);

proveri("rezultat je nova instanca", [k4 !== dodata, k4 !== uklonjena], [true, true]);

proveri("transakcije se dele, ne kopiraju", k4.transakcije[0] === dodata.transakcije[0], true);

// ── 4 · istorija kao lanac referenci na prethodnu knjigu ──────────────
naslov("4 · istorija");

proveri("nova knjiga pamti onu od koje je nastala",
        [k4.prethodna === k3, k3.prethodna === k2, k0.prethodna], [true, true, null]);

proveri("celokupna istorija: pet verzija, od prazne do tekuće", istorija(k4).length, 5);

proveri("stanje svake verzije", istorija(k4).map(k => k.stanje), [0, 120000, 75000, 57000, 57000]);

proveri("istorija druge knjige je nezavisna", istorija(vega).length, 4);

### Šta ove provere zapravo dokazuju

**„transakcije se dele, ne kopiraju"** — `k4.transakcije[0] === dodata.transakcije[0]` je `true`. Nova verzija
**nije** duboka kopija: nosi iste zamrznute transakcije. To je ono što trajnu strukturu čini upotrebljivom —
da se kopira sve, cena bi rasla sa svakom izmenom.

**„geter transakcije vraća kopiju"** — `push` na dobijeni niz prolazi (niz je običan), ali knjigu ne dodiruje.
Bez getera koji kopira, `Object.freeze(knjiga)` bi bio nedovoljan: zamrznut je objekat, ne i niz iza polja.

**„stanje je izvedeno"** — jedina provera koja bi pukla da se `stanje` čuva kao polje: `dodata.stanje` je `61000`
odmah po nastanku, bez ijednog koraka ažuriranja.

In [ ]:
// ── 5 · pretraga ──────────────────────────────────────────────────────
naslov("5 · pretraga");

proveri("dva kriterijuma", pretraga([realizovane, naTeret], k4).map(x => x.opis), ["Zakup prostora", "Struja"]);

proveri("parametrizovan kriterijum", pretraga([preko(50000)], k4).map(x => x.opis), ["Uplata kupca", "Nabavka robe"]);

proveri("kriterijum po mesecu i statusu",
        pretraga([uMesecu("2026-08"), realizovane], k4).map(x => x.opis),
        ["Uplata kupca", "Zakup prostora", "Struja"]);

proveri("kriterijum po broju računa", pretraga([saRacuna("265-0002")], k4).map(x => x.opis), ["Nabavka robe"]);

proveri("prazan niz kriterijuma vraća sve", pretraga([], k4).length, 4);

proveri("pretraga ne dira knjigu", k4.transakcije.length, 4);

// dokaz da se kriterijumi primenjuju REDOM kojim su navedeni:
// prvi vidi sve transakcije, svaki sledeći samo one koje su prošle prethodni
const brojac = { A: 0, B: 0 };
const A = t => (brojac.A++, realizovane(t));
const B = t => (brojac.B++, naTeret(t));

pretraga([A, B], k4);
const redosledAB = [brojac.A, brojac.B];

brojac.A = 0; brojac.B = 0;
pretraga([B, A], k4);
const redosledBA = [brojac.A, brojac.B];

console.log("broj poziva [A, B]:", redosledAB, "| broj poziva [B, A]:", redosledBA);

proveri("prvi navedeni kriterijum se primenjuje prvi (vidi sve, sledeći samo preostale)",
        [redosledAB, redosledBA], [[4, 3], [3, 4]]);

proveri("redosled menja tok primene, ali ne i konačan skup",
        pretraga([A, B], k4).map(x => x.opis), pretraga([B, A], k4).map(x => x.opis));

// niz kriterijuma se može sastaviti i u toku rada — npr. iz popunjenih polja obrasca
const popunjeno = { status: "realizovana", tip: "na teret", minIznos: 10000 };
const izPolja = [
    ...(popunjeno.status   ? [t => t.status === popunjeno.status] : []),
    ...(popunjeno.tip      ? [t => t.tip === popunjeno.tip]       : []),
    ...(popunjeno.minIznos ? [preko(popunjeno.minIznos)]          : [])
];
proveri("niz kriterijuma sastavljen u toku rada",
        pretraga(izPolja, k4).map(x => x.opis), ["Zakup prostora", "Struja"]);

### Kako se dokazuje „redom kojim su navedeni"

Predikati su čiste funkcije, pa se po **rezultatu** redosled ne vidi: `[A, B]` i `[B, A]` daju isti skup.
Vidi se po **broju poziva**. Zato su `A` i `B` omotani brojačem: prvi navedeni predikat pozvan je 4 puta
(sve transakcije), drugi 3 puta (samo one koje su prošle prvi). Zamenom mesta brojevi se zamene.

Ovo je i praktična posledica: kriterijum koji najviše sužava treba navesti prvi, jer smanjuje posao svima posle njega.

**Zapis `(brojac.A++, realizovane(t))`** koristi zarez-operator: izvrši se levi izraz zbog efekta, a vrati se
desni. To je namerno **nečista** funkcija, napravljena samo da bi se izmerio redosled — jedina takva u fajlu.

In [ ]:
// ── 6 · opoziv ────────────────────────────────────────────────────────
naslov("6 · opoziv");

const opozvana = opoziv(k4, 2);                          // vraćamo se na verziju sa indeksom 2

console.log("verzija 2 je sadržala:", istorija(k4)[2].transakcije.map(x => x.opis));
console.log("opozvana sadrži:", opozvana.transakcije.map(x => `${x.opis}(${x.status})`));

proveri("sadrži sve transakcije zadate verzije, plus kasnije kao stornirane",
        opozvana.transakcije.map(x => `${x.opis}(${x.status})`),
        ["Uplata kupca(realizovana)", "Zakup prostora(realizovana)",
         "Struja(stornirana)", "Nabavka robe(stornirana)"]);

proveri("stanje opozvane jednako je stanju verzije na koju se vraćamo",
        [opozvana.stanje, istorija(k4)[2].stanje], [75000, 75000]);

proveri("opozvana se nastavlja na knjigu nad kojom je opoziv pozvan",
        [opozvana.prethodna === k4, istorija(opozvana).length], [true, 6]);

proveri("opoziv na indeks 0 stornira sve", opoziv(k4, 0).transakcije.map(x => x.status),
        ["stornirana", "stornirana", "stornirana", "stornirana"]);

proveri("nepostojeći indeks se odbija", greskaOd(() => opoziv(k4, 99)) !== null, true);

// ── 7 · opoziv ne menja nijednu knjigu ni njihove transakcije ─────────
naslov("7 · opoziv ništa ne mutira");

proveri("tekuća knjiga netaknuta",
        [k4.transakcije.length, k4.stanje, k4.transakcije.map(x => x.status)],
        [4, 57000, ["realizovana", "realizovana", "realizovana", "nerealizovana"]]);

proveri("nijedna knjiga iz istorije nije izmenjena",
        istorija(k4).map(k => k.transakcije.length), [0, 1, 2, 3, 4]);

proveri("nijedna verzija ne sadrži storniranu transakciju",
        istorija(k4).every(k => k.transakcije.every(x => x.status !== "stornirana")), true);

proveri("originalni objekti transakcija netaknuti",
        [t3.status, t4.status], ["realizovana", "nerealizovana"]);

proveri("stornirane u opozvanoj su KOPIJE, ne isti objekti",
        [opozvana.transakcije[2] !== t3, opozvana.transakcije[2].redniBroj === t3.redniBroj], [true, true]);

proveri("pretraga nad opozvanom vidi stornirane",
        pretraga([stornirane], opozvana).map(x => x.opis), ["Struja", "Nabavka robe"]);

// ── izveštaj ──────────────────────────────────────────────────────────
console.log(`\n═══ ${prosle}/${ukupno} provera prošlo ═══` +
            (prosle === ukupno ? " sve ispravno" : " IMA GREŠAKA"));

---
# Pitanja uz `k1v2.html`

Jedanaest pitanja postavljenih u komentarima uz rešenje. Odgovori su ovde, sa pokretljivim primerima
za one gde tvrdnja nije očigledna. Zajednički imenilac svih jedanaest izdvojen je u poslednjem
odeljku — **Koncepti**.

## P1 · Zašto `const` + IIFE — može li `new` sa `let` unutra

`const` nije suština; on samo brani da neko kasnije zameni ime `Transakcija`. Suština je **gde živi zapis
u kome stoji brojač**. IIFE se izvrši **jednom**, pa postoji **jedan** zapis koji dele svi pozivi fabrike.

Zapažanje da je `let` u konstruktoru nedostupan spolja je tačno — ali beskorisno, jer `new` pravi
**nov zapis pri svakom pozivu**, pa se brojač resetuje. Dobilo bi se privatno stanje **po instanci**,
a traži se **statičko** — jedno za ceo tip.

Sa `new` ipak može, ali drugim sredstvom — pravim statičkim privatnim poljem `static #brojac`.
Fabrika je izabrana jer je predmet funkcionalni, ne zato što je klasa nemoguća.

## P2 · Šta biva sa `redniBroj` u kopiji

Kopira se **vrednost**, ne pristup. `redniBroj` je broj, pa kopija dobija svoje mesto sa istim brojem;
veze sa originalom nema (`kopija === original` je `false`).

Ali formulacija „može da se kopira, ali ne može da se menja" nije sasvim tačna. Zamrznuta je *postojeća*
kopija. Ništa ne brani da se napravi kopija sa **drugim** brojem — `{ ...t, redniBroj: 999 }` prolazi.
To što `stornirana` zadržava redni broj je **odluka koda**, a ne osobina jezika.

In [ ]:
// ── P1 · gde živi brojač ─────────────────────────────────────────────

function TransakcijaNew(opis) {
    let sledeciRedniBroj = 1;              // NOV zapis pri svakom new
    this.opis = opis;
    this.redniBroj = sledeciRedniBroj++;
}
const pNew1 = new TransakcijaNew("a"), pNew2 = new TransakcijaNew("b");
console.log("new + let unutra →", [pNew1.redniBroj, pNew2.redniBroj], "— resetuje se");
console.log("pNew1.sledeciRedniBroj:", pNew1.sledeciRedniBroj, "— nedostupno, ali i beskorisno");

class TransakcijaKlasa {                   // pravi statički privatni atribut
    static #brojac = 1;
    constructor(opis) { this.opis = opis; this.redniBroj = TransakcijaKlasa.#brojac++; }
}
const pKlasa1 = new TransakcijaKlasa("a"), pKlasa2 = new TransakcijaKlasa("b");
console.log("class + static # →", [pKlasa1.redniBroj, pKlasa2.redniBroj], "— radi");

// ── P2 · vrednost se kopira, veza se ne prenosi ──────────────────────

const pKopija = stornirana(t3);
console.log("\nt3.redniBroj:", t3.redniBroj, "| kopija.redniBroj:", pKopija.redniBroj);
console.log("isti objekat:", pKopija === t3, "| isti status:", pKopija.status === t3.status);
console.log("spread NE brani drugi broj:", Object.freeze({ ...t3, redniBroj: 999 }).redniBroj,
            "— dogovor, ne zabrana");

## P3 · Šta ako je neko polje složena vrednost

Ovo je najozbiljnija zamka u celom rešenju. `{ ...t }` i `Object.freeze` su **oba plitka** — staju na
prvom nivou. Ako bi neko polje bilo niz ili objekat, kopija i original bi pokazivali na **isti** ugnježđeni
objekat, pa bi izmena kroz kopiju bila vidljiva kroz original. Zamrznut je omotač, ne ono na šta polje pokazuje.

Cela garancija „`opoziv` ne menja nijednu transakciju" tada bi pala.

Tri izlaza: ručno kopirati ugnježđeno (`{ ...t, stavke: [...t.stavke] }`), duboko zamrznuti pri nastanku,
ili napraviti duboku kopiju sa `structuredClone`. U ovom zadatku problem **ne postoji** jer su sva polja
transakcije proste vrednosti — i baš zato je plitki spread ovde dovoljan.

## P4 · Zašto bi metoda morala da zove fabriku

Ne mora. Metoda **sme** sama da napravi kopiju spread-om; tada nema poziva fabrike, pa se brojač ne pomera.
Prigovor iz Jedinice 2 važi samo za varijantu koja rekonstruiše kroz fabriku.

Ali iskaču dva nova problema, i zavise od toga odakle metoda čita podatke:

- **strelica nad zapisom poziva** — kopija nasledi i metodu, ali ta metoda i dalje gleda u **stari** objekat;
- **metoda sa `this`** — radi ispravno kad se zove kao `objekat.metoda()`, ali kad se prosledi kao vrednost,
  `this` je `undefined`, a `{ ...undefined }` je `{}` — polja **tiho nestanu**, bez ijedne greške.

Na pitanje o referenci: **ne**, kopija ne drži nikakvu vezu ka originalu. Ako se hoće, mora se izričito
dopisati: `{ ...t, status: "stornirana", nastalaOd: t }`.

In [2]:
// ── P3 · spread i freeze su PLITKI ───────────────────────────────────

const pSlozena1 = Object.freeze({ redniBroj: 1, stavke: ["a", "b"], meta: { odobrio: "Pera" } });
const pSlozena2 = Object.freeze({ ...pSlozena1, status: "stornirana" });

console.log("niz je ISTI objekat u obe:  ", pSlozena1.stavke === pSlozena2.stavke);
console.log("objekat je ISTI u obe:      ", pSlozena1.meta === pSlozena2.meta);

pSlozena2.stavke.push("c");                 // izmena kroz KOPIJU
console.log('Da li je isti?', pSlozena1.stavke === pSlozena2.stavke)
pSlozena2.meta.odobrio = "Mika";

console.log("original.stavke posle toga: ", pSlozena1.stavke);
console.log("original.meta posle toga:   ", pSlozena1.meta);
console.log("isFrozen(original):", Object.isFrozen(pSlozena1),
            "| isFrozen(original.stavke):", Object.isFrozen(pSlozena1.stavke));

const dubokoZamrzni = o => {
    Object.values(o).forEach(v => { if (v && typeof v === "object") dubokoZamrzni(v); });
    return Object.freeze(o);
};
const pDuboko = dubokoZamrzni({ redniBroj: 1, stavke: ["a"] });
try { pDuboko.stavke.push("x"); } catch (g) { console.log("duboko zamrznuto →", g.constructor.name); }

// ── P4 · metoda koja sama pravi kopiju ───────────────────────────────

const SaStrelicom = (opis, status) => {
    const o = Object.freeze({
        opis, status,
        storniraj: () => Object.freeze({ ...o, status: "stornirana" })   // gleda u o iz zapisa
    });
    return o;
};
const pStrel = SaStrelicom("Struja", "realizovana");
console.log("\nstrelica → kopija.status:", pStrel.storniraj().status);
console.log("   poziv metode NAD KOPIJOM daje opis:", pStrel.storniraj().storniraj().opis,
            "— i dalje gleda u stari o");

const SaThis = (opis, status) => Object.freeze({
    opis, status,
    storniraj() { return Object.freeze({ ...this, status: "stornirana" }); }
});
const pThis = SaThis("Struja", "realizovana");
console.log("this → ispravan poziv:", pThis.storniraj());
console.log("this → prosleđena kao vrednost:", [pThis].map(pThis.storniraj),
            "← opis NESTAO, bez greške");
console.log("kopija drži vezu ka originalu:", pThis.storniraj() === pThis);

niz je ISTI objekat u obe:   true
objekat je ISTI u obe:       true
Da li je isti? true
original.stavke posle toga:  [ "a", "b", "c" ]
original.meta posle toga:    { odobrio: "Mika" }
isFrozen(original): true | isFrozen(original.stavke): false
duboko zamrznuto → TypeError

strelica → kopija.status: stornirana
   poziv metode NAD KOPIJOM daje opis: Struja — i dalje gleda u stari o
this → ispravan poziv: {
  opis: "Struja",
  status: "stornirana",
  storniraj: [Function: storniraj]
}
this → prosleđena kao vrednost: [ { status: "stornirana" } ] ← opis NESTAO, bez greške
kopija drži vezu ka originalu: false


## P5 · Zašto su `naziv`, `maticniBroj`, `pib` i `prethodna` u prvom argumentu

`Object.defineProperties(objekat, deskriptori)` — prvi argument je objekat koji se gradi, drugi je mapa
opisa svojstava.

`naziv`, `maticniBroj`, `pib` i `prethodna` su **uskladištene vrednosti**: ništa se ne računa, pa idu u
običan literal. `transakcije` i `stanje` **moraju** da budu geteri — prvi da pri svakom čitanju vrati kopiju,
drugi da preračuna iz transakcija.

Zato `enumerable: true` nije ukras: geter definisan preko `defineProperties` podrazumevano **nije nabrojiv**,
pa se bez toga ne bi video u `console.log`, `Object.keys` ni u spread-u — radio bi samo kad mu se izričito
priđe tačkom.

**Šta se gubi:** `prethodna` je javno polje, pa svako može da prošeta ceo lanac unazad. To je ovde namerno
(`istorija` na tome i stoji), ali znači da istorija nije skrivena.

## P6 · Kako se pristupa istoriji

Da — peti argument se preslikava na parametar `prethodna`:

```js
GlavnaKnjiga(naziv, maticniBroj, pib, sa(t, tr), knjiga)
//                                               ↑ postaje prethodna nove knjige
```

Istorija se dobija ponovljenim koracima `.prethodna`, što `istorija` radi rekurzivno.

Suptilno: `knjiga` se pominje **unutar** izraza koji tek definiše `knjiga`. Radi zato što se strelica ne
**izvršava** odmah — kad je neko kasnije pozove, ime je odavno vezano.

## P7 · Skoro tačno, uz jednu ispravku

`preko(5000)` ne dobija niz nego **jednu transakciju**:

```js
preko(5000)                      // → t => t.iznos > 5000
preko(5000)(jednaTransakcija)    // → true / false
```

Niz mu se nikad ne prosleđuje — `filter` ga zove po jednom za svaki element:

```js
pretraga([preko(5000)], knjiga)
// →  [preko(5000)].reduce((niz, k) => niz.filter(k), knjiga.transakcije)
// →  knjiga.transakcije.filter(preko(5000))
// →  filter interno zove preko(5000)(t) za svako t
```

## P8 · `verzije[redniBrojUIstoriji]` nije poziv funkcije

`verzije` je **niz**, vezan u redu iznad: `const verzije = istorija(knjiga);`.
Uglaste zagrade su pristup po indeksu, ne poziv.

```js
f(x)      // poziv funkcije — okrugle zagrade
a[i]      // element niza na indeksu i — uglaste zagrade
```

Znači: „element niza `verzije` na indeksu `redniBrojUIstoriji`".

In [ ]:
// ── P5 · zašto enumerable: true ──────────────────────────────────────

const pBezEnum = Object.freeze(Object.defineProperties({ a: 1 }, { b: { get: () => 2 } }));
const pSaEnum  = Object.freeze(Object.defineProperties({ a: 1 }, { c: { get: () => 3, enumerable: true } }));

console.log("bez enumerable → Object.keys:", Object.keys(pBezEnum), "| ali pBezEnum.b radi:", pBezEnum.b);
console.log("sa  enumerable → Object.keys:", Object.keys(pSaEnum),  "| pSaEnum.c:", pSaEnum.c);
console.log("spread bez enumerable:", { ...pBezEnum }, "| sa enumerable:", { ...pSaEnum });

// u pravoj knjizi: stanje i transakcije se vide upravo zato što je enumerable postavljen
console.log("Object.keys(k4):", Object.keys(k4));

## P9 · `Set` nije zbog jedinstvenosti

Zapažanje da su redni brojevi već jedinstveni je tačno — `new Set(...)` ovde ne izbacuje ništa.
Tu je zbog **brzine**: `.has()` je O(1), dok bi `.includes()` nad nizom bio O(n) po svakoj proveri.

Drugi filter je za **drugu vrstu** ponavljanja: isti objekat transakcije nalazi se u **više kasnijih verzija**
(i verzija 3 i verzija 4 sadrže `t3`), pa ga `flatMap` izbaci više puta. Bez njega bi `Struja` bila
stornirana dvaput u istoj knjizi.

## P10 · Šta prosleđujemo funkciji `stornirana`

Ne prosleđujemo joj ništa — prosleđujemo **nju samu** kao vrednost. `map` je zatim zove po jednom za svaki
element. Bitno: `map` šalje **tri** argumenta — element, indeks i ceo niz. `stornirana` deklariše samo jedan
parametar, pa se ostala dva ignorišu; zato je bezbedno. Klasična zamka kad funkcija prima više parametara je
`["1","2","3"].map(parseInt)`.

## P11 · Zašto se prosleđuje cela knjiga, a ne objekat koji samo pamti istoriju

Prvi razlog je doslovan — tekst zadatka to traži: *„istoriju izmena u vidu reference na prethodnu glavnu
knjigu"*.

Drugi je bolji. Svaka knjiga je **već** nepromenljiv, potpun snimak trenutka. Pokazivanje na nju košta jednu
referencu, a daje ceo tadašnji sadržaj — uključujući izvedeno `stanje`, besplatno. Objekat koji bi „samo
pamtio istoriju" morao bi da čuva isto to, dakle bio bi glavna knjiga pod drugim imenom.

`opoziv` upravo to i koristi: `stara.transakcije`. Da je `prethodna` dnevnik izmena, morao bi da ga premota
unazad da rekonstruiše stanje.

**Cena:** ceo lanac ostaje u memoriji dok postoji tekuća knjiga, i istorija je javna.

In [ ]:
// ── P9 · zašto Set, i zašto DRUGI filter ─────────────────────────────

const pVerzije = istorija(k4);
console.log("verzija ukupno:", pVerzije.length);

console.log("ista transakcija u v3 i v4:", pVerzije[3].transakcije[2] === pVerzije[4].transakcije[2],
            "— isti OBJEKAT, ne kopija");

const pPosle = pVerzije.slice(3).flatMap(k => k.transakcije);
console.log("flatMap verzija posle indeksa 2:", pPosle.map(x => x.redniBroj), "— ponavljanja");

const pBileRanije = new Set(pVerzije[2].transakcije.map(x => x.redniBroj));
const pKorak1 = pPosle.filter(x => !pBileRanije.has(x.redniBroj));
console.log("posle prvog filtera (Set):     ", pKorak1.map(x => x.redniBroj), "— 3 se javlja dvaput");

const pKorak2 = pKorak1.filter((x, i, n) => n.findIndex(y => y.redniBroj === x.redniBroj) === i);
console.log("posle drugog filtera:          ", pKorak2.map(x => x.redniBroj));

// ── P10 · šta map prosleđuje ─────────────────────────────────────────

const pGledaj = (...args) => args.length;
console.log("\nmap šalje ovoliko argumenata:", ["a", "b"].map(pGledaj));
console.log("klasična zamka, map(parseInt):", ["1", "2", "3"].map(parseInt));
console.log("map(stornirana) je bezbedan jer prima 1 parametar:",
            [t3].map(stornirana).map(x => x.status));

---
# Rečnik pojmova iz ovog zadatka

| Pojam | Značenje | Gde se vidi |
| --- | --- | --- |
| **čista funkcija** | rezultat zavisi samo od argumenata; poziv ne ostavlja trag | `sa`, `bez`, `stanjeOd`, `istorija`, `pretraga`, `opoziv` |
| **nepromenljivost** | vrednost se ne menja, nego se zamenjuje novom | `Object.freeze` svuda; `[...niz, x]` umesto `push` |
| **zatvorenje** | funkcija nosi pokazivač na zapis poziva u kome je nastala (mehanizam objašnjen u Digresiji) | brojač u `Transakcija`, niz `t` u `GlavnaKnjiga`, granica u `preko` |
| **funkcija višeg reda** | prima ili vraća funkciju | `pretraga`, `preko`, `uMesecu`, `map(stornirana)` |
| **predikat** | funkcija koja vraća `true`/`false` | svi kriterijumi pretrage |
| **parcijalna primena** | deo argumenata sada, ostatak kasnije | `preko(50000)` daje predikat |
| **preklapanje (fold)** | svođenje niza na jednu vrednost | `stanjeOd`, `pretraga`, gradnja knjige `vega` |
| **trajna struktura** | izmena pravi novu verziju, stara i dalje važi | lanac `prethodna` |
| **izvedena vrednost** | računa se iz izvora umesto da se čuva | geter `stanje` |
| **referencijalna transparentnost** | poziv se sme zameniti svojim rezultatom | sve osim `Transakcija` (brojač) i `proveri` (ispis) |
| **efekti na ivici** | nečisto se skuplja na jedno mesto | `console.log` samo u DEO II |
| **fabrička funkcija** | funkcija koja vraća objekat, bez `new` i `this` | `Transakcija`, `GlavnaKnjiga` |

# Šta zadatak zapravo proverava

1. **Da li se privatnost može napraviti bez klase** — zatvorenje umesto `#` polja (jedinica 1).
2. **Da li se izmena može izraziti bez mutacije** — nova instanca umesto `push` (jedinice 2 i 3).
3. **Da li se istorija može dobiti besplatno** — kao posledica gradnje, a ne kao poseban dnevnik (jedinice 3 i 4).
4. **Da li se funkcije koriste kao podaci** — kriterijumi koji se prosleđuju, slažu i sastavljaju u toku rada (jedinica 5).
5. **Da li se garancija „ne menja original" oslanja na jezik ili na pažnju** — `freeze` + kopije (jedinica 6).

# Veza sa Kolokvijumom II

Dva mesta u ovom rešenju rešava mehanizam sa K2:

- **`pretraga` pravi `n` međunizova** za `n` kriterijuma. Transdjuseri spajaju sve filtere u **jedan** prolaz,
  bez ijednog međuniza.
- **Redni broj kao promenljivo stanje** može se zameniti beskonačnim generatorom `brojac()`, čime `Transakcija`
  postaje čista, a izvor rednih brojeva postaje običan tok vrednosti.

Oba su pokazana u `k1kaok2/k1_kao_k2.ipynb`.

---
# Koncepti

Jedanaest pitanja iz `k1v2.html` deluju kao jedanaest različitih nedoumica, ali ispod njih stoje
**četiri** ideje. Svaka se javlja u više pitanja, i skoro sve nedoumice nestaju kad se te četiri razdvoje.

| | Koncept | Pitanja | Koliko puta |
| --- | --- | --- | --- |
| 1 | vrednost naspram reference — šta se kopira, a šta deli | P2, P3, P4, P9, P11 | 5 |
| 2 | funkcija kao vrednost — prosleđivanje naspram pozivanja | P4, P7, P8, P10 | 4 |
| 3 | odakle funkcija čita podatke | P1, P4, P6 | 3 |
| 4 | kad se šta izvršava | P1, P5, P6 | 3 |

Primetno je da se **P4 javlja u tri od četiri** — zato i deluje najzamršenije: u njemu se sastaju
kopiranje, prosleđivanje funkcije i izvor podataka.

---

## 1 · Vrednost naspram reference

**Šta se dešava.** Prosta vrednost (broj, string, `true`, `null`) se pri kopiranju **umnožava** — kopija
dobija svoje mesto sa istim sadržajem. Složena vrednost (objekat, niz, funkcija) se **ne umnožava**;
kopira se samo pokazivač, pa oba imena vode na **isti** objekat.

`{ ...t }` i `Object.freeze` rade tačno **jedan nivo duboko**. Zamrznut je omotač, a ne ono na šta polja
pokazuju.

**Gde se to javilo:**

- **P2** — `redniBroj` je broj, pa je u kopiji nov primerak. Nema „pristupa" originalu, samo isti sadržaj.
- **P3** — da je polje niz, kopija i original bi delili taj niz i izmena kroz jedno videla bi se kroz drugo.
  Ovo je opasna strana.
- **P4** — kopija napravljena spread-om **ne** drži vezu ka originalu; ako se veza želi, mora se dopisati.
- **P9** — ista transakcija je **isti objekat** u više verzija knjige, pa je `flatMap` izbaci više puta.
  Otud drugi filter.
- **P11** — a to isto deljenje je korisna strana: nova verzija knjige ne kopira transakcije, samo pokazuje
  na iste. Otud je istorija jeftina.

**Pitanje koje razrešava:** *da li je ovo isti objekat, ili samo isti sadržaj?* Odgovara `===`.

---

## 2 · Funkcija kao vrednost

**Šta se dešava.** Ime funkcije bez zagrada je **vrednost** — može se dodeliti, staviti u niz, proslediti.
Tek okrugle zagrade je **pozivaju**. Zbog toga je važno kod svakog izraza znati ko koga zove i sa čim.

**Gde se to javilo:**

- **P10** — `map(stornirana)` **prosleđuje** funkciju; `map` je zatim zove po jednom za svaki element,
  i to sa **tri** argumenta (element, indeks, niz).
- **P7** — `preko(5000)` je poziv koji vraća **funkciju**. Ta funkcija prima **jednu transakciju**, ne niz;
  poziva je `filter`, jednu po jednu.
- **P8** — obrnuta zabuna: `verzije[i]` **nije** poziv. Okrugle zagrade zovu, uglaste indeksiraju.
- **P4** — metoda prosleđena kao vrednost prestaje da bude metoda; gubi `this` i tiho vraća pogrešno.

**Pitanje koje razrešava:** *gde su zagrade i ko ih stavlja?*

```js
f          // vrednost — funkcija se ne izvršava
f(x)       // poziv sa x
f(x)(y)    // f vraća funkciju, pa se i ona zove
a[i]       // indeks, ne poziv
niz.map(f) // f se ne zove ovde — map je zove kasnije, sam
```

---

## 3 · Odakle funkcija čita podatke

**Šta se dešava.** Funkcija ima **tri** moguća izvora, i oni se ponašaju potpuno različito:

| Izvor | Kad se određuje | Da li preživi prosleđivanje |
| --- | --- | --- |
| **parametri** | pri svakom pozivu | da — stižu kao argumenti |
| **`this`** | zavisi od **načina** poziva | ne — `objekat.metoda` izvučena kao vrednost gubi `this` |
| **zapis poziva u kome je napisana** | pri nastanku funkcije, zauvek | da — pokazivač se nosi sa funkcijom |

**Gde se to javilo:**

- **P1** — brojač dolazi iz zapisa. Pitanje „jednom ili po pozivu" je pitanje **koliko zapisa** postoji.
- **P6** — strelica pominje `knjiga`, ime iz zapisa u kome je napisana.
- **P4** — cela razlika između dve varijante `storniraj` je izvor: strelica čita iz zapisa (i zato gleda
  u stari objekat), a `this` varijanta čita iz načina poziva (i zato puca kad se prosledi).

**Pitanje koje razrešava:** *koja imena funkcija pominje, a nisu joj parametri?* To je isto pravilo iz
Digresije, samo primenjeno unazad.

---

## 4 · Kad se šta izvršava

**Šta se dešava.** Tri različita trenutka se lako pomešaju: kad je kod **napisan**, kad je izraz
**izvršen**, i kad se funkcija **pozove**. Vrednost se računa u trenutku izvršavanja, ne pisanja.

**Gde se to javilo:**

- **P1** — IIFE se izvrši **jednom**, pa je zapis jedan. `new` se izvršava **po pozivu**, pa je zapisa
  onoliko koliko ima instanci. Isti kod, različit broj izvršavanja.
- **P6** — strelica pominje `knjiga` pre nego što `knjiga` postoji. Radi zato što se **ne izvršava tada**;
  do prvog poziva ime je odavno vezano.
- **P5** — geter se ne izračunava pri pravljenju objekta nego pri **svakom čitanju**. Zato `stanje` ne može
  da se raziđe sa sadržajem — nema uskladištene vrednosti koja bi zastarela.

**Pitanje koje razrešava:** *u kom trenutku se ovaj izraz stvarno izvršava — sada, ili kad ga neko pozove?*

---

## Četiri pitanja umesto jedanaest

Kad se naiđe na nejasan red koda, ova četiri pitanja pokrivaju skoro sve što se gore javilo:

1. **Da li je ovo isti objekat ili samo isti sadržaj?** → vrednost naspram reference
2. **Gde su zagrade i ko ih stavlja?** → prosleđivanje naspram pozivanja
3. **Koja imena pominje, a nisu mu parametri?** → izvor podataka
4. **Kad se ovo stvarno izvršava?** → trenutak izvršavanja

## Šta nije bilo problem

Vredi primetiti i šta se **nije** javilo ni u jednom pitanju: nepromenljivost sama po sebi, `reduce`,
`filter`, rekurzija, predikati. Ti delovi su prošli bez zapinjanja.

Sve četiri nedoumice tiču se **JavaScript-a kao jezika** — kako čuva vrednosti, kako zove funkcije, odakle
im podaci i kada šta radi — a ne funkcionalne paradigme. To je i očekivano: paradigma je skup odluka
*šta pisati*, a ova četiri koncepta određuju *šta se pri tome zaista dešava*.